# Stage 3 - DPO Preference Alignment
### GenAI / Agentic AI Domain Assistant &middot; Base model: `Qwen2.5-0.5B` &middot; Framework: [Unsloth](https://github.com/unslothai/unsloth) + [TRL](https://github.com/huggingface/trl)

Pipeline: Base Model &rarr; Stage 1: Non-Instruction FT &rarr; Stage 2: Instruction FT &rarr; **Stage 3: DPO Alignment**

This notebook:
1. Loads the Stage 2 SFT model (`models/sft_adapter/`)
2. Formats `data/preference_dataset.jsonl` (50 chosen/rejected pairs) for `DPOTrainer`
3. Runs DPO alignment (Unsloth-patched, PEFT-native reference model - no second full model needed)
4. Saves the final model to `models/dpo_adapter/`
5. Runs the full three-way comparison (Base vs. SFT vs. DPO) &rarr; writes `reports/final_evaluation.md`

> Run on a free Colab/Kaggle T4 GPU. Total runtime: ~5-10 minutes for this 0.5B model.


In [ ]:
%%capture
# Colab: Runtime > Change runtime type > T4 GPU, then run this cell first.
!pip install -q -U unsloth unsloth_zoo
!pip install -q -U "trl>=0.12.0" peft accelerate bitsandbytes datasets sentencepiece


In [ ]:
import os

REPO_URL = "https://github.com/Abhishek15016/prepminds.git"

if not os.path.exists("data") and REPO_URL:
    repo_name = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
    if not os.path.exists(repo_name):
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(repo_name)

assert os.path.exists("data"), "Could not find data/ - clone/open the repo first."
os.makedirs("models", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
import sys
sys.path.insert(0, os.getcwd())


In [ ]:
from src.eval_questions import EVAL_QUESTIONS, clean_completion, judge_pair, markdown_table

print(f"Loaded {len(EVAL_QUESTIONS)} canonical evaluation questions (same set used in every stage).")


## Step 0 - (Optional) Hugging Face Hub setup

Lets this notebook automatically pull the Stage 2 (SFT) adapter from the Hub if it isn't on local disk, and push the final DPO adapter when done.

In [ ]:
from huggingface_hub import login
import os

# --- Hugging Face Hub config -------------------------------------------------
# Colab wipes local disk between sessions, so each notebook can optionally push
# its adapter to the Hub, and the *next* notebook will automatically pull it
# back down if no local copy is found. Set your HF username below and
# (recommended) add a Colab secret named HF_TOKEN with a "write" token -
# the key icon in the left sidebar - so login() doesn't need to prompt you.
HF_USERNAME = "abhishek15016"  # leave blank to skip the Hub entirely
PUSH_TO_HUB = bool(HF_USERNAME)
PRIVATE_REPO = True
HF_MODEL_PREFIX = "qwen2.5-0.5b-genai-agentic"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if PUSH_TO_HUB:
    login(token=HF_TOKEN or None)  # prompts interactively if HF_TOKEN isn't set

STAGE1_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage1-non-instruction" if HF_USERNAME else ""
STAGE2_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage2-sft" if HF_USERNAME else ""
STAGE3_HF_REPO = f"{HF_USERNAME}/{HF_MODEL_PREFIX}-stage3-dpo" if HF_USERNAME else ""

def push_adapter(model, tokenizer, repo_id, stage_name):
    if not (PUSH_TO_HUB and repo_id):
        print(f"Skipping Hub push for {stage_name} (set HF_USERNAME above to enable).")
        return
    model.push_to_hub(repo_id, token=HF_TOKEN or None, private=PRIVATE_REPO)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN or None, private=PRIVATE_REPO)
    print(f"Pushed {stage_name} adapter to https://huggingface.co/{repo_id}")


## Step A - Load the Stage 2 (SFT) model and patch TRL's DPOTrainer for Unsloth

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
import torch

PatchDPOTrainer()

MAX_SEQ_LENGTH = 2048
BASE_MODEL_NAME = "unsloth/Qwen2.5-0.5B-bnb-4bit"
SFT_LOCAL_DIR = "models/sft_adapter"

if os.path.exists(SFT_LOCAL_DIR):
    SFT_SOURCE = SFT_LOCAL_DIR
elif STAGE2_HF_REPO:
    SFT_SOURCE = STAGE2_HF_REPO
    print(f"No local SFT adapter - pulling from the Hub: {STAGE2_HF_REPO}")
else:
    raise FileNotFoundError(
        "No local SFT adapter at models/sft_adapter and no STAGE2_HF_REPO "
        "configured. Run instruction_finetuning.ipynb first, or set HF_USERNAME "
        "above to the account that adapter was pushed to."
    )
print("Loading from:", SFT_SOURCE)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_SOURCE,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

if not hasattr(model, "peft_config"):
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
model.print_trainable_parameters()


## Step B - Load and format the preference dataset

`data/preference_dataset.jsonl` stores `prompt` / `chosen` / `rejected` as chat-message lists. `DPOTrainer` wants plain strings, so we render the prompt with the chat template (with a generation prompt) and take the raw assistant text for chosen/rejected.

In [ ]:
from datasets import load_dataset

pref_ds = load_dataset("json", data_files="data/preference_dataset.jsonl", split="train")
print(f"{len(pref_ds)} preference pairs")

def format_dpo(example):
    prompt_text = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False, add_generation_prompt=True
    )
    return {
        "prompt": prompt_text,
        "chosen": example["chosen"][0]["content"],
        "rejected": example["rejected"][0]["content"],
    }

pref_ds = pref_ds.map(format_dpo, remove_columns=pref_ds.column_names)
print(pref_ds[0]["prompt"][-300:])
print("--- chosen ---")
print(pref_ds[0]["chosen"][:300])
print("--- rejected ---")
print(pref_ds[0]["rejected"][:300])


## Step C - Configure and run DPO

In [ ]:
from trl import DPOConfig, DPOTrainer
from src.trl_compat import build_config

dpo_config = build_config(
    DPOConfig,
    beta=0.1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    num_train_epochs=2,
    learning_rate=5e-6,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    lr_scheduler_type="linear",
    seed=3407,
    max_length=1024,
    max_prompt_length=512,
    output_dir="outputs/stage3_dpo",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # PEFT model -> TRL uses the base weights (adapters disabled) as the implicit reference
    args=dpo_config,
    train_dataset=pref_ds,
    tokenizer=tokenizer,
)


In [ ]:
# Take a screenshot of this cell's output for your README ("Training screenshots or logs").
dpo_result = dpo_trainer.train()
dpo_result.metrics


## Step D - Save the final DPO-aligned model

In [ ]:
SAVE_DIR = "models/dpo_adapter"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved final DPO-aligned adapter to", SAVE_DIR)
print("This is the model src/inference.py loads by default.")


In [ ]:
push_adapter(model, tokenizer, STAGE3_HF_REPO, "Stage 3 (DPO, final model)")
if PUSH_TO_HUB and STAGE3_HF_REPO:
    print(f"\nsrc/inference.py can now also load this model directly from the Hub:")
    print(f'  python src/inference.py -q "What is LoRA?" --model {STAGE3_HF_REPO}')


## Step E - Final three-way evaluation: Base vs. SFT vs. DPO (Step 10 of the assignment)

In [ ]:
FastLanguageModel.for_inference(model)

def generate_chat(active_model, active_tokenizer, question, max_new_tokens=220):
    convo = [
        {"role": "system", "content": "You are a friendly expert tutor in Generative AI and Agentic AI. Explain concepts in depth with intuitive analogies, practical production insight, and interview-ready takeaways, in a clear and engaging style."},
        {"role": "user", "content": question},
    ]
    # Tokenize directly through apply_chat_template (tokenize=True) instead of
    # rendering to a string and re-tokenizing it separately - re-tokenizing a
    # rendered template string can produce a different token count than what
    # was actually fed to generate(), which then makes the prompt-length slice
    # below cut into the prompt itself instead of the new tokens (the model
    # appearing to "echo" the question/system prompt is exactly that bug).
    input_ids = active_tokenizer.apply_chat_template(
        convo, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(active_model.device)
    out = active_model.generate(
        input_ids=input_ids, max_new_tokens=max_new_tokens, do_sample=True,
        temperature=0.7, top_p=0.9, pad_token_id=active_tokenizer.eos_token_id,
    )
    text = active_tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

dpo_answers = [generate_chat(model, tokenizer, q) for q in EVAL_QUESTIONS]
for q, a in zip(EVAL_QUESTIONS, dpo_answers):
    print("Q:", q)
    print("A (DPO):", a[:300])
    print("-" * 80)


In [ ]:
# Fresh instances of the base model and the SFT model for a clean 3-way, paired comparison.
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

def generate_raw(question, max_new_tokens=180):
    prompt = f"Question: {question}\nAnswer:"
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    out = base_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,
        top_p=0.9, repetition_penalty=1.3, pad_token_id=base_tokenizer.eos_token_id,
    )
    text = base_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return clean_completion(text, question)

base_answers = [generate_raw(q) for q in EVAL_QUESTIONS]
del base_model
torch.cuda.empty_cache()

sft_model, sft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_SOURCE, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True,
)
sft_tokenizer = get_chat_template(sft_tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(sft_model)
sft_answers = [generate_chat(sft_model, sft_tokenizer, q) for q in EVAL_QUESTIONS]
del sft_model
torch.cuda.empty_cache()


In [ ]:
rows = []
for q, base_a, sft_a, dpo_a in zip(EVAL_QUESTIONS, base_answers, sft_answers, dpo_answers):
    _, sft_vs_base_reason = judge_pair(q, "SFT", sft_a, "SFT", sft_a)
    winner, reason = judge_pair(q, "SFT", sft_a, "DPO", dpo_a)
    rows.append((q, base_a, sft_a, dpo_a, winner, reason))

table = markdown_table(
    ["Question", "Base Model Answer", "SFT Model Answer", "DPO Model Answer", "Best Answer", "Reason"],
    rows,
)

report = f"""# Final Evaluation: Base vs. SFT vs. DPO-Aligned Model

**Base model:** `{BASE_MODEL_NAME}`
**SFT model:** Stage 1 + Stage 2 (instruction fine-tuned on `data/instruction_dataset.jsonl`)
**DPO model:** Stage 1 + Stage 2 + Stage 3 (DPO-aligned on `data/preference_dataset.jsonl`) - **final model**

{table}

## Evaluation criteria
Correctness, helpfulness, domain accuracy, safety, tone, clarity,
hallucination reduction, professional response quality (per assignment Step 10).

## Summary
Non-instruction fine-tuning adapts the base model's vocabulary and style to
the GenAI/Agentic-AI domain. Instruction fine-tuning teaches it to actually
answer questions rather than free-complete text. DPO alignment then sharpens
*which* correct-shaped answer it prefers, pushing it toward the more
complete, professional, and domain-precise response style captured in the
preference dataset's "chosen" examples and away from the shallow, generic,
or unsafe style of the "rejected" examples.

*Auto-generated by `notebooks/dpo_alignment.ipynb`. The "Best Answer" /
"Reason" columns are a heuristic first draft (see `src/eval_questions.py`) -
review before submission.*
"""

with open("reports/final_evaluation.md", "w", encoding="utf-8") as f:
    f.write(report)

print(report)


---
### Done - final model saved to `models/dpo_adapter/` (and to `STAGE3_HF_REPO` on the Hub, if configured)

Run `src/inference.py` (or import `generate_answer` from it) to query the final,
DPO-aligned domain assistant interactively - it accepts either the local
`models/dpo_adapter` path or a Hugging Face Hub repo id via `--model`.